# Introduction


In this assignment, you will practice building and training Convolutional Neural Networks with Pytorch to solve computer vision tasks.  This assignment includes two sections, each involving different tasks:

(1) Image Classification. Predict image-level category labels on two historically notable image datasets: **CIFAR-10** and **MNIST**.

(2) Image Segmentation. Predict pixel-wise classification (semantic segmentation) on synthetic input images formed by superimposing MNIST images on top of CIFAR images.

You will design your own models in each section and build the entire training/testing pipeline with PyTorch. 
PyTorch provides optimized implementations of the building blocks and additional utilities, both of which will be necessary for experiments on real datasets. It is highly recommended to read the official [documentation](https://pytorch.org/docs/stable/index.html) and [examples](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html) before starting your implementation. There are some APIs that you'll find useful:
[Layers](http://pytorch.org/docs/stable/nn.html),
[Activations](https://pytorch.org/docs/stable/nn.html#non-linear-activations-weighted-sum-nonlinearity),
[Loss functions](http://pytorch.org/docs/stable/nn.html#loss-functions),
[Optimizers](http://pytorch.org/docs/stable/optim.html)

It is highly recommended to use Google Colab and run the notebook on a GPU node. Check https://colab.research.google.com/ and look for tutorials online. To use a GPU go to Runtime -> Change runtime type and select GPU. 






# (1) Image Classification

In this section, you will design and train an image classification network, which takes images as input and outputs vectors whose length equals the number of possible categories on **MNIST** and **CIFAR-10** datasets. 

You can design your models by borrowing ideas from recent architectures, e.g., ResNet, but you may not simply copy an entire existing model. 

For image classification, you can use a built-in dataset provided by [torchvision](https://pytorch.org/vision/stable/index.html), a PyTorch official extension for image tasks. 

To finish this section step by step, you need to:

* Prepare data by building a dataset and dataloader. (with [torchvision](https://pytorch.org/vision/stable/index.html))

* Implement training code (6 points) & testing code (6 points), including model saving and loading.

* Construct a model (12 points) and choose an optimizer (3 points).

* Describe what you did, any additional features you implemented, and/or any graphs you made in training and evaluating your network. Also report final test accuracy @100 epochs in a writeup: hw3.pdf (3 points)

In [1]:
import numpy as np
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import sampler
import torchvision
import torchvision.transforms as T


## Data Preparation:

Setup a Dataset for training and testing.

Datasets load single training examples one a time, so we practically wrap each Dataset in a DataLoader, which loads a data batch in parallel.

We provide an example for setting up a training set for MNIST, and you should complete the rest. 

In [2]:
mnist_train = torchvision.datasets.MNIST('./data', train = True, transform = T.functional.to_tensor, download = True)
mnist_test = torchvision.datasets.MNIST('./data', train = False, transform = T.functional.to_tensor, download = True)
mnist_loader_train = torch.utils.data.DataLoader(mnist_train,
                                          batch_size=128,
                                          shuffle=True,
                                          num_workers=0)
mnist_loader_test = torch.utils.data.DataLoader(mnist_test,
                                          batch_size=128,
                                          num_workers=0)

cifar_train = torchvision.datasets.CIFAR10('./data', train = True, transform = T.functional.to_tensor, download = True)
cifar_test = torchvision.datasets.CIFAR10('./data', train = False, transform = T.functional.to_tensor, download = True)

cifar10_loader_train = torch.utils.data.DataLoader(cifar_train,
                                          batch_size=128,
                                          shuffle=True,
                                          num_workers=0)
cifar10_loader_test = torch.utils.data.DataLoader(cifar_test,
                                          batch_size=128,
                                          num_workers=0)


## Design/choose your own model structure (12 points) and optimizer (3 points).
You might want to adjust the following configurations for better performance:

(1) Network architecture:
- You can borrow some ideas from existing CNN designs, e.g., ResNet where
the input from the previous layer is added to the output
https://arxiv.org/abs/1512.03385
- Note: Do not **directly copy** an entire existing network design.

(2) Architecture hyperparameters:
- Filter size, number of filters, and number of layers (depth). Make careful choices to tradeoff computational efficiency and accuracy.
- Pooling vs. Strided Convolution
- Batch normalization
- Choice of non-linear activation

(3) Choice of optimizer (e.g., SGD, Adam, Adagrad, RMSprop) and associated hyperparameters (e.g., learning rate, momentum).

In [ ]:
#Basic model, feel free to customize the layout to fit your model design.

class myNet(nn.Module):
    def __init__(self):
        super(myNet, self).__init__()

        # Set up your own CNN.

    def forward(self, x):
        # forward
        return out

class MNIST_models(myNet):
    def __init__(self):
        super(MNIST_models, self).__init__()
        self.num_classes = 10
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(16 * 14 * 14, 128)
        self.fc2 = nn.Linear(128, self.num_classes)

    def save(self):
        torch.save(self.state_dict(), "mnist_model.pth")
    
    def load(self, device = "cpu"):
        self.load_state_dict(torch.load("mnist_model.pth", map_location=device))
        return self


    def forward(self, x):
        # forward
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        out = self.fc2(x)
        return out
    
class CIFAR10_models(myNet):
    def __init__(self):
        super(CIFAR10_models, self).__init__()
        self.num_classes = 10
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 2 * 2, 128)
        self.fc2 = nn.Linear(128, self.num_classes)
        self.model_save_path = "./model"

    def save(self):
        torch.save(self.state_dict(), "cifar10_model.pth")
    def load(self, device = "cpu"):
        self.load_state_dict(torch.load("cifar10_model.pth", map_location=device))
        return self

    def forward(self, x):
        # forward
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)
        x = F.relu(self.conv4(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
    
        x = F.relu(self.fc1(x))
        out = self.fc2(x)
        return out



## Training (6 points)

Train a model on the given dataset using the PyTorch Module API.

Inputs:
- loader_train: The loader from which train samples will be drawn from.
- loader_test: The loader from which test samples will be drawn from.
- model: A PyTorch Module giving the model to train.
- optimizer: An Optimizer object we will use to train the model.
- epochs: (Optional) A Python integer giving the number of epochs to train for.

Returns: Nothing, but prints model accuracies during training.

In [ ]:
def train(loader_train, loader_test, model, optimizer, epochs=50):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss() # choose your loss here, if needed
    
    for e in range(epochs):
        model.train()
        for t, (x, y) in enumerate(loader_train):
            x = x.to(device)
            y = y.to(device)
            results = model(x)
            loss = criterion(results, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if t % 100 == 0:
                print('Epoch %d, Iteration %d, loss = %.4f' % (e, t, loss.item()))
        test(loader_test, model)

## Testing (6 points)
Test a model using the PyTorch Module API.

Inputs:
- loader: The loader from which test samples will be drawn from.
- model: A PyTorch Module giving the model to test.

Returns: Nothing, but prints model accuracies during training.

In [ ]:
def test(loader, model):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_correct = 0
    num_samples = 0
    model.eval() # set model to evaluation mode
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            scores = model(x)
            calc = scores.argmax(dim=1)
            for i in range(len(calc)):
                if calc[i] == y[i]:
                    num_correct += 1
            num_samples += calc.size(0)
        acc = float(num_correct) / num_samples
        print('Eval %d / %d correct (%.2f)' % (num_correct, num_samples, 100 * acc))


Describe your design details in the writeup hw3.pdf. (3 points)

Finish your model and optimizer below.

In [ ]:
lr = 1e-2 
momentum = 0.9
weight_decay = 1e-4

model = CIFAR10_models()
optimizer = optim.SGD(model.parameters(), lr, momentum, weight_decay)
train(cifar10_loader_train, cifar10_loader_test, model, optimizer, epochs=100)
model.save()

In [ ]:
model = MNIST_models()
optimizer = optim.SGD(model.parameters(), lr, momentum, weight_decay)
train(mnist_loader_train, mnist_loader_test, model, optimizer, epochs=10)
model.save()